# Challenge 5: Baseline vs candidate, on quality, cost and latency

**EDD: Change and Compare. Model upgrade framework, step 5: run the experiment.** This is a model migration in miniature: the same agent, prompt, dataset and evaluators on two models.
Baseline is Claude Haiku 4.5, candidate is Claude Sonnet 4.6. A better score is only half the question; the other half is what it costs and how long it takes,
so you'll compare **quality, cost and latency together, by segment**, because an average hides where a candidate regresses.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import json
import os
import sys
import warnings


for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client, uuid7

import lab_links
import lab_sdk

client = Client()
PROD = lab_sdk.prod_project()
GOLDEN = lab_sdk.ds_golden()
print("prod project:", PROD)
print("golden dataset:", GOLDEN)

from concurrent.futures import ThreadPoolExecutor

from eval_lab import metrics
from eval_lab.agent import SYSTEM_PROMPT_V1, SYSTEM_PROMPT_V2, build_agent
from eval_lab.data import EXTERNAL_RESULTS
from eval_lab.evaluators import (EVALUATOR_VERSION, answer_quality, exact_trajectory,
                                 keyword_match, make_run_agent)
from utils.models import BASELINE_ID, CANDIDATE_ID, candidate_model, model

## 1. Run both models on the test split

Both experiments run at the same time on the golden `test` split (12 examples, four per segment), with the same prompt (`v2`) and the same three evaluators.
Challenge 8's gate reuses these two experiments, so run this once.

In [ ]:
def run_experiment(prefix, llm, model_id, system_prompt=SYSTEM_PROMPT_V2, prompt_label="v2"):
    return client.evaluate(
        make_run_agent(build_agent(system_prompt, llm=llm)),
        data=client.list_examples(dataset_name=GOLDEN, splits=["test"]),
        evaluators=[exact_trajectory, keyword_match, answer_quality],
        experiment_prefix=prefix,
        metadata={"model": model_id, "prompt": prompt_label, "evaluators": EVALUATOR_VERSION},
        max_concurrency=4,
    )


with ThreadPoolExecutor(max_workers=2) as pool:
    f_base = pool.submit(run_experiment, "baseline", model, BASELINE_ID)
    f_cand = pool.submit(run_experiment, "candidate", candidate_model, CANDIDATE_ID)
    baseline, candidate = f_base.result(), f_cand.result()
print("baseline :", baseline.experiment_name)
print("candidate:", candidate.experiment_name)

## 2. What did each run cost?

Every run records its token counts (`run.prompt_tokens`, `run.completion_tokens`). Cost is tokens times a price. The table below is an **assumption for this lab**
(approximate list prices, USD per 1M tokens). Replace it with your contract rates before using any cost figure for a real decision.

In [ ]:
PRICES = metrics.PRICES
for model_id, price in PRICES.items():
    print(f"{model_id:48} input ${price['input']:.2f}  output ${price['output']:.2f}  per 1M tokens")

Write `run_cost(run, model_id)`: the cost in USD of one run, from its prompt and completion tokens (treat a missing count as 0) and the prices for that model.
A reference solution is in the next cell.

In [ ]:
# Reference solution: try writing this yourself first.
def run_cost(run, model_id):
    price = PRICES[model_id]
    return ((run.prompt_tokens or 0) * price["input"] + (run.completion_tokens or 0) * price["output"]) / 1_000_000

In [ ]:
sample = next(iter(client.list_runs(project_name=baseline.experiment_name, is_root=True)))
cost = run_cost(sample, BASELINE_ID)
assert cost >= 0, "run_cost must return a non-negative number"
print(f"one baseline run: {sample.prompt_tokens} prompt + {sample.completion_tokens} completion tokens = ${cost:.5f}")
from lab_state import mark_done
mark_done("/tmp/.sdk-ch5-cost-ok")

## 3. Compare by segment

`metrics.experiment_rows` joins every run to its example's `segment` (through `reference_example_id`) and returns quality, cost, latency, errors and policy violations for each run.
`metrics.summarize` turns those into the table. The result is written to `comparison.json`.

In [ ]:
import pandas as pd

segs = metrics.example_segments(client, GOLDEN)
summary = {
    "baseline": metrics.summarize(metrics.experiment_rows(client, baseline.experiment_name, BASELINE_ID, segs, cost_fn=run_cost)),
    "candidate": metrics.summarize(metrics.experiment_rows(client, candidate.experiment_name, CANDIDATE_ID, segs, cost_fn=run_cost)),
}

os.makedirs(lab_sdk.EXPORT_DIR, exist_ok=True)
with open(f"{lab_sdk.EXPORT_DIR}/comparison.json", "w") as f:
    json.dump({"prices_are_an_assumption": metrics.PRICES, **summary}, f, indent=2)

table = []
for name, s in summary.items():
    for seg, m in [("ALL", s["overall"]), *s["segments"].items()]:
        table.append({"segment": seg, "model": name, "n": m["n"], "quality": m["quality"],
                      "cost/example $": m["cost_per_example"], "quality per $": m["quality_per_dollar"],
                      "p50 s": m["latency_p50"], "p95 s": m["latency_p95"], "error rate": m["error_rate"]})
pd.DataFrame(table).sort_values(["segment", "model"]).round(4).reset_index(drop=True)

Read it the way a decision-maker would: is the candidate better where it matters (look at each segment, not only `ALL`), and is the quality gain worth the extra cost and latency?
Quality per dollar puts the two on one scale. Challenge 8 turns this reading into a rule.

## 4. Pairwise comparison

`evaluate_comparative` takes two experiments and an evaluator that sees both runs for each example and returns a score per run. Here the preference is code: the run whose
tool trajectory matches the reference wins. An LLM judge fits the same shape. It needs no new model calls.

In [ ]:
from langsmith.evaluation import evaluate_comparative


def prefer_exact(runs, example):
    scores = {run.id: int((run.outputs or {}).get("trajectory") == example.outputs["trajectory"]) for run in runs}
    return {"key": "prefer_exact", "scores": scores}


try:
    pairwise = evaluate_comparative(
        (baseline.experiment_name, candidate.experiment_name),
        evaluators=[prefer_exact],
        experiment_prefix="pairwise",
    )
    print("pairwise experiment:", pairwise.experiment_name)
except Exception as exc:
    print("Pairwise step skipped:", type(exc).__name__, "- open the two experiments in LangSmith and use Compare instead.")

## 5. Results from outside LangSmith

Your agent may run somewhere else (a batch job, another team's service). You can still put its results in LangSmith: create an experiment tied to the dataset with
`client.create_project(..., reference_dataset_id=...)`, then add one run per example with `client.create_run(..., reference_example_id=...)`, including the error for the one that failed.
`EXTERNAL_RESULTS` holds four results, the last one an error. Run the cell and open the new experiment in LangSmith.

In [ ]:
from datetime import datetime, timezone

EXT = f"external-upload-{lab_sdk.sfx()}"
golden = client.read_dataset(dataset_name=GOLDEN)
examples = list(client.list_examples(dataset_name=GOLDEN, splits=["test"]))[:4]
uploaded = []
if lab_sdk.experiments(client, golden.id, "external-upload"):
    print(f"{EXT} already exists; skipping the upload")
else:
    client.create_project(project_name=EXT, reference_dataset_id=golden.id)
    for example, row in zip(examples, EXTERNAL_RESULTS):
        run_id = uuid7()
        now = datetime.now(timezone.utc)
        client.create_run(
            id=run_id, name="external_agent", run_type="chain", inputs=example.inputs,
            outputs=None if row.get("error") else {"answer": row["answer"], "trajectory": row["trajectory"]},
            error=row.get("error"), project_name=EXT, reference_example_id=example.id,
            start_time=now, end_time=now,
        )
        uploaded.append((run_id, row))
    client.flush()
    ext_id = client.read_project(project_name=EXT).id
    for run_id, row in uploaded:
        lab_sdk.feedback_when_ready(client, run_id, session_id=ext_id, key="exact_trajectory", score=row["score"],
                                    comment="scored outside LangSmith")
    client.flush()
    print(f"Uploaded {len(uploaded)} runs to {EXT} and attached their scores")

## 6. Prompt versions

Prompts are versioned like data. Push the two system prompts to your workspace, list the commits, pull the first one back, and evaluate it, so a prompt change gets a score before it ships.
This runs one more experiment (a couple of minutes) and needs prompt access in your workspace; if your plan has no prompt access the cell says so and carries on.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

name = lab_sdk.prompt_name()
try:
    for label, text in (("v1", SYSTEM_PROMPT_V1), ("v2", SYSTEM_PROMPT_V2)):
        template = ChatPromptTemplate.from_messages([("system", text), ("placeholder", "{messages}")])
        client.push_prompt(name, object=template, commit_description=f"order-support system prompt {label}")
    commits = sorted(client.list_prompt_commits(name), key=lambda c: c.created_at)
    print(len(commits), "commits:", [c.commit_hash[:8] for c in commits])

    oldest = client.pull_prompt(f"{name}:{commits[0].commit_hash}")
    old_text = oldest.messages[0].prompt.template
    prompt_v1 = run_experiment("prompt-v1", model, BASELINE_ID, system_prompt=old_text, prompt_label="v1")
    print("prompt-v1 experiment:", prompt_v1.experiment_name)
except Exception as exc:
    print("Prompt step skipped:", type(exc).__name__, "- push prompts from the LangSmith UI instead.")

In the LangSmith UI, open **Prompts**, choose `order-support-...`, and open a commit in the **Playground** to try a change against a dataset before committing it.

## 7. See it in LangSmith

In [ ]:
lab_links.dataset_link(GOLDEN, hint="Experiments tab: select baseline-... and candidate-... and click Compare; the pairwise and external-upload experiments are listed too")